In [ ]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import pyproj

torch.set_printoptions(sci_mode = False)

# Antarctica's hidden world - Antarctica's Gamburtsev Province Project (AGAP)

- Data from Bedmap2: https://ramadda.data.bas.ac.uk/repository/entry/show?entryid=synth%3A2fd95199-365e-4da1-ae26-3b6d48b3e6ac%3AL0JBU18yMDA3X0FHQVBfQUlSX0JNMi5jc3Y%3D
- Collected during polar year 2007 (rather 2008/2009)

- Load 2.5 M values
- skip first 18 rows in loading

Data:
- 66 flight trajectories
- Around 20 meter spacing between data points

In [ ]:
bas_agab = pd.read_csv("~/data/bedmap/BAS_2007_AGAP_AIR_BM2.csv", skiprows = range(0, 18))
# on_bad_lines = "skip"

In [ ]:
bas_agab

In [ ]:
bas_agab['trajectory_id'].value_counts() # 66
bas_agab['trace_number'].value_counts() # useless

bas_agab['date'].value_counts()
bas_agab['surface_altitude (m)'].value_counts()
bas_agab['bedrock_altitude (m)'].value_counts() # around 5000 values missing

bas_agab['two_way_travel_time (m)'].value_counts() # useless
bas_agab['aircraft_altitude (m)'].value_counts() # useless
bas_agab['along_track_distance (m)'].value_counts() # useless

In [ ]:
# Remove use rows
bas_agab = bas_agab.drop(columns = ['trace_number', 
                                    'two_way_travel_time (m)',
                                    'aircraft_altitude (m)',
                                    'along_track_distance (m)',
                                    # Also drop date and time for now
                                    'date',
                                    'time_UTC'])

# Remove rows with no valid bedrock altitude value
bas_agab = bas_agab[bas_agab['bedrock_altitude (m)'] != - 9999.0]
# Remaining entries all have surface s, thickness h, and bed b 

In [ ]:
bas_agab = bas_agab.rename(columns = {'land_ice_thickness (m)': 'h', 
                           'surface_altitude (m)': 's',
                           'bedrock_altitude (m)': 'b'
                           })

In [ ]:
# This does not excatly add up
bas_agab["s"] - bas_agab["h"] - bas_agab["b"]

In [ ]:
fig = px.histogram(x = (bas_agab["s"] - bas_agab["h"] - bas_agab["b"]))
fig.show()
# From firn or rounding inconsitencies?

In [ ]:
# Once really high value
np.max(bas_agab["s"] - bas_agab["h"] - bas_agab["b"])
np.argmax(bas_agab["s"] - bas_agab["h"] - bas_agab["b"])

In [ ]:
bas_agab.loc[[1720548]]
# surprising.

In [ ]:
bas_agab

# Convert to Polar Stereographic

In [ ]:
# This is how projections work. check through tool
polarstereo_to_lonlat = pyproj.Transformer.from_crs(crs_from = pyproj.CRS("epsg:4326"),  
                                                    crs_to = pyproj.CRS("epsg:3031"),
                                                    always_xy = True) # xy order convention

# Pass in lon, lat and return x, y in polar stereo
x_array, y_array = polarstereo_to_lonlat.transform(
    bas_agab["longitude (degree_east)"], 
    bas_agab["latitude (degree_north)"])

bas_agab["x"] = x_array
bas_agab["y"] = y_array

# Find closest point / grid cell

# Find trajectory over Dome A (and associates flight line)

In [ ]:
# max surface: Dome Argus
bas_agab[bas_agab["s"] == np.max(bas_agab["s"])]

In [ ]:
# max bedrock:
bas_agab[bas_agab["b"] == np.max(bas_agab["b"])]

In [ ]:
bas_agab[bas_agab["trajectory_id"] == 11]

In [ ]:
bas_agab[bas_agab["trajectory_id"] == 18]

In [ ]:
fig = go.Figure(data = go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11]["y"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [ ]:
fig = go.Figure(data = go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 900000]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 900000]["y"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [ ]:
flight_11_transect = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 1000000][bas_agab["x"] <= 1050000]
flight_11_transect_small = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 1032000][bas_agab["x"] <= 1035000]

In [ ]:
flight_11_transect.tail(n = 2)[["x", "y"]].astype(str)
# 4 meter y rise over 20 meters x
flight_11_transect.iloc[[400, 401, 402]].astype("str")

In [ ]:
fig = go.Figure(data = go.Scatter(x = flight_11_transect["x"], 
                                  y = flight_11_transect["b"], 
                                  mode = 'markers'))
fig.add_trace(go.Scatter(x = flight_11_transect["x"], 
                                  y = flight_11_transect["s"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [ ]:
fig = go.Figure(data = go.Scatter(x = flight_11_transect_small["x"], 
                                  y = flight_11_transect_small["b"], 
                                  mode = 'markers'))
fig.add_trace(go.Scatter(x = flight_11_transect_small["x"], 
                                  y = flight_11_transect_small["s"], 
                                  mode = 'markers'))
fig.add_vline(x = 1032000, line_width = 1, line_color = "gray")
fig.add_vline(x = 1032500, line_width = 1, line_color = "gray")
fig.add_vline(x = 1033000, line_width = 1, line_color = "gray")
fig.add_vline(x = 1033500, line_width = 1, line_color = "gray")
fig.add_vline(x = 1034000, line_width = 1, line_color = "gray")
fig.add_vline(x = 1034500, line_width = 1, line_color = "gray")
fig.add_vline(x = 1035000, line_width = 1, line_color = "gray")
fig.update_layout(title = "Flight line 11 zoomed in with 500 m spacing", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [ ]:
flight_11_transect_small["b"]

In [ ]:
# n = 138 
# std = 107.0, Normalized by N-0 by default
# var = 11,400.0
np.std(flight_11_transect_small["b"])

In [ ]:
np.std(flight_11_transect_small["b"])

In [ ]:
# Bed and height have an almost perfect negative correlation (complement definition)
np.corrcoef(flight_11_transect_small["b"], flight_11_transect_small["h"])

# Surprisingly the bed and  surface have a pretty strong negative relationship: Because surface height is capped?
np.corrcoef(flight_11_transect_small["b"], flight_11_transect_small["s"])

In [ ]:
# Covariance matrix has variance, variance and covariance
np.cov(flight_11_transect_small["h"], flight_11_transect_small["s"]).astype("str")

# Look at roughness ratio between surface and bed

In [ ]:
fig = go.Figure(data = go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11]["b"], 
                                  mode = 'markers'))
fig.add_trace(go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11]["s"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()